# 03 矩阵分解（SVD via scipy / sklearn）

无需 C 编译器，直接用 `sklearn.decomposition.TruncatedSVD` 实现隐因子矩阵分解推荐。

> **论文中写法**：采用 SVD 对用户-商品交互矩阵进行低秩近似，
> 学习用户隐因子 U 和商品隐因子 V，通过 Uᵢ · Vⱼ 预测评分。

In [1]:
import pandas as pd
import numpy as np
import pickle
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split

ratings = pd.read_csv('../data/csv/ratings.csv')
print(f'评分数: {len(ratings)}')

评分数: 18080


In [2]:
# ── 构建稀疏评分矩阵 ─────────────────────────────────────────
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42)

all_users    = sorted(ratings.user_id.unique())
all_products = sorted(ratings.product_id.unique())
u2i = {u: i for i, u in enumerate(all_users)}
p2i = {p: i for i, p in enumerate(all_products)}
i2u = {i: u for u, i in u2i.items()}
i2p = {i: p for p, i in p2i.items()}

rows = train_df.user_id.map(u2i)
cols = train_df.product_id.map(p2i)
R = csr_matrix((train_df.rating.values, (rows, cols)),
               shape=(len(all_users), len(all_products)), dtype=np.float32)
print('评分矩阵:', R.shape, '  稀疏度:', 1 - R.nnz / (R.shape[0]*R.shape[1]))

评分矩阵: (206, 496)   稀疏度: 0.8584403382398997


In [3]:
# ── SVD 训练 （k 个隐因子）─────────────────────────────────
K = 50   # 隐因子维度（论文中调参分析此超参数）
svd = TruncatedSVD(n_components=K, random_state=42)
U   = svd.fit_transform(R)           # (n_users, K)  用户隐因子矩阵
Vt  = svd.components_                # (K, n_items)  商品隐因子矩阵
print(f'U shape: {U.shape}, Vt shape: {Vt.shape}')

# 重构评分矩阵
R_pred = np.dot(U, Vt)               # (n_users, n_items)
print(f'预测矩阵: {R_pred.shape}')

U shape: (206, 50), Vt shape: (50, 496)
预测矩阵: (206, 496)


In [4]:
# ── RMSE 评估 ────────────────────────────────────────────────
test_users = test_df.user_id.map(u2i)
test_items = test_df.product_id.map(p2i)
pred_vals  = R_pred[test_users, test_items]
true_vals  = test_df.rating.values

rmse = np.sqrt(np.mean((pred_vals - true_vals)**2))
mae  = np.mean(np.abs(pred_vals - true_vals))
print(f'SVD (k={K})  RMSE={rmse:.4f}  MAE={mae:.4f}')

SVD (k=50)  RMSE=1.5823  MAE=1.4649


In [5]:
# ── Precision@K / Recall@K / NDCG@K ─────────────────────────
def precision_recall_ndcg_at_k(recommended, relevant, k=10):
    rec_k = recommended[:k]
    hits  = len(set(rec_k) & set(relevant))
    p = hits / k
    r = hits / len(relevant) if relevant else 0
    dcg  = sum(1/np.log2(i+2) for i, iid in enumerate(rec_k) if iid in set(relevant))
    idcg = sum(1/np.log2(i+2) for i in range(min(k, len(relevant))))
    return p, r, (dcg/idcg if idcg > 0 else 0)

# 用户已交互商品（训练集）
user_interacted = train_df.groupby('user_id')['product_id'].apply(set).to_dict()
# 测试集相关商品
user_relevant   = test_df.groupby('user_id')['product_id'].apply(list).to_dict()

ps, rs, ns = [], [], []
for uid in list(user_relevant.keys())[:150]:
    if uid not in u2i: continue
    ui = u2i[uid]
    scores = R_pred[ui].copy()
    # 过滤已交互
    for pid in user_interacted.get(uid, set()):
        if pid in p2i: scores[p2i[pid]] = -999
    top_idx  = np.argsort(scores)[::-1][:20]
    top_pids = [i2p[i] for i in top_idx]
    p, r, n  = precision_recall_ndcg_at_k(top_pids, user_relevant[uid], k=10)
    ps.append(p); rs.append(r); ns.append(n)

print(f'SVD  Precision@10={np.mean(ps):.4f}  Recall@10={np.mean(rs):.4f}  NDCG@10={np.mean(ns):.4f}')

SVD  Precision@10=0.0420  Recall@10=0.0235  NDCG@10=0.0405


In [6]:
# ── 不同 K 值消融实验（论文用表格）────────────────────────────
results = []
for k in [10, 20, 50, 100]:
    svd_k = TruncatedSVD(n_components=k, random_state=42)
    U_k   = svd_k.fit_transform(R)
    Rp    = np.dot(U_k, svd_k.components_)
    rm    = np.sqrt(np.mean((Rp[test_users, test_items] - true_vals)**2))
    results.append({'K': k, 'RMSE': round(rm, 4), '方差解释率': round(svd_k.explained_variance_ratio_.sum(), 4)})

import pandas as pd
print(pd.DataFrame(results).to_string(index=False))

  K   RMSE  方差解释率
 10 1.4911 0.1109
 20 1.5232 0.2159
 50 1.5823 0.4702
100 1.6412 0.7526


In [7]:
# ── 保存 SVD 推荐结果 ─────────────────────────────────────────
svd_results = {}
for ui, uid in i2u.items():
    scores = R_pred[ui].copy()
    for pid in user_interacted.get(uid, set()):
        if pid in p2i: scores[p2i[pid]] = -999
    top_idx = np.argsort(scores)[::-1][:50]
    svd_results[uid] = [i2p[i] for i in top_idx]

with open('../data/csv/svd_results.pkl', 'wb') as f:
    pickle.dump(svd_results, f)
print('SVD 推荐结果已保存 (data/csv/svd_results.pkl)')

SVD 推荐结果已保存 (data/csv/svd_results.pkl)
